In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

import sys
src_path = str(Path.cwd().parent.parent)
if src_path not in sys.path:
    sys.path.append(src_path)
import src.d00_utils.utilities as utils
import src.d00_utils.dirnames as dn

from bioio import BioImage
import bioio_ome_tiff
import bioio_tifffile
from bioio.writers import OmeTiffWriter

from tqdm import tqdm


In [ ]:
input_dirpath = Path(input())

In [ ]:
proc_dirpath = utils.get_proc_dirpath(input_dirpath)
caax_dirpath = proc_dirpath / 'caax'
caax_dirpath.mkdir(exist_ok=True)

cell_dirpath = proc_dirpath / 'cell'
cell_dirpath.mkdir(exist_ok=True)

imgpaths = [path for path in input_dirpath.glob('*.ome.tif')]
imgpaths.sort()

cellch = 2
caaxch = 1

for imgpath in tqdm(imgpaths):
    imgname = imgpath.name

    cellpath = cell_dirpath / imgname
    
    # open image
    if not cellpath.exists():
        img_file = BioImage(imgpath, reader=bioio_ome_tiff.Reader)
        img = img_file.data
            
        utils.create_ch_subset([cellch], imgpath, img=img, img_file=img_file, output_dir=cell_dirpath)
        utils.create_ch_subset([caaxch], imgpath, img=img, img_file=img_file, output_dir=caax_dirpath)
        
    else:
        print(f'{imgname} does not exist in this directory.')
        
                

## Before moving on the next step, run FIJI bleach correction on single channel images

In [ ]:
cell_dirpath = Path(input())

In [ ]:
caax_dirpath = Path(input())

In [ ]:
input_dirpath = Path(input())

In [ ]:
proc_dirpath = utils.get_proc_dirpath(caax_dirpath)

# use FIJI bleach correction, then stitch images together prior to alignment
caax_corr_dirpath = caax_dirpath / 'corrbleach_ometif'
cell_corr_dirpath = cell_dirpath / 'corrbleach_ometif'
caax_cell_actin_dirpath = proc_dirpath / 'caax_cell_actin'
caax_cell_actin_dirpath.mkdir(exist_ok=True)

caax_cell_dirpath = proc_dirpath / 'caax_cell'
caax_cell_dirpath.mkdir(exist_ok=True)

imgpaths = [p for p in caax_corr_dirpath.glob('*.ome.tif')]

actinch = 0

for i, imgpath in enumerate(tqdm(imgpaths)):

        imgname = imgpath.name
        caax_corr_path = caax_corr_dirpath / imgname
        cell_corr_path = cell_corr_dirpath / imgname
        orig_img_path = input_dirpath / imgname

        caax_corr = BioImage(caax_corr_path, reader=bioio_tifffile.Reader).data
        cell_corr = BioImage(cell_corr_path, reader=bioio_tifffile.Reader).data
        img_file = BioImage(orig_img_path, reader=bioio_ome_tiff.Reader)
        # actin = img_file.data[:, actinch, np.newaxis, :, :, :]

        # caax_cell_actin_stack = np.concatenate([caax_corr, cell_corr, actin], axis=1)
        # ome_metadata = utils.construct_ome_metadata(caax_cell_actin_stack, img_file)
        # OmeTiffWriter.save(caax_cell_actin_stack, caax_cell_actin_dirpath / imgname, ome_xml=ome_metadata)

        caax_cell_stack = np.concatenate([caax_corr, cell_corr], axis=1)
        ome_metadata = utils.construct_ome_metadata(caax_cell_stack, img_file)
        OmeTiffWriter.save(caax_cell_stack, caax_cell_dirpath / imgname, ome_xml=ome_metadata)
        